In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)


import gc
import ast
import torch
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset
from peft import PeftModel
from peft import LoraConfig
from peft import get_peft_model
from transformers import Trainer
from transformers import TrainingArguments
from transformers import Blip2Processor, Blip2ForConditionalGeneration

In [ ]:
train_img_root = './blip/data/train'
test_img_root = './blip/data/test'
normal_img_root = './blip/data/normal'

# 1. 基本加载
df = pd.read_csv('./blip/data/processed_dataset.csv')
df['caption'] = df['raw'].apply(ast.literal_eval)  # 将字符串转换为列表
# 2. 根据 'split' 列进行划分
train_df = df[df['split'] == 'train']
# print(train_df.head())  # 查看前5行
test_df = df[df['split'] == 'test']
# print(test_df.head())  # 查看前5行
normal_df = df[df['split'] == 'normal']

train_data = []

for i in range(len(train_df)):
    img_path = train_df.iloc[i]['filename']
    caption = train_df.iloc[i]['caption']
    for cap in caption[0:1]:
        train_data.append({
            "image": os.path.join(train_img_root, img_path),
            "text": cap.lower()
        })
print(train_data)
print("Length of train_data:", len(train_data))

test_data = []

for i in range(len(test_df)):
    img_path = test_df.iloc[i]['filename']
    caption = test_df.iloc[i]['caption']
    for cap in caption[0:1]:
        test_data.append({
            "image": os.path.join(test_img_root, img_path),
            "text": cap.lower()
        })
print(test_data) 
print("Length of test_data:", len(test_data))

normal_data = []
for i in range(len(normal_df)):
    img_path = normal_df.iloc[i]['filename']
    caption = normal_df.iloc[i]['caption']
    for cap in caption[0:1]:
        normal_data.append({
            "image": os.path.join(normal_img_root, img_path),
            "text": cap.lower()
        })
print(normal_data)
print("Length of normal_data:", len(normal_data))

In [ ]:
model_name = "Salesforce/blip2-flan-t5-xl"
cache_dir = "./model/blip/blip2_cache"
dtype = torch.bfloat16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model
processor = Blip2Processor.from_pretrained(
    model_name,
    cache_dir=cache_dir
    )
base = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    torch_dtype=dtype,
    ).to(device)


model = PeftModel.from_pretrained(base, "./blip/blip2_lora")
# model = base
model.eval()  # 设置为评估模式

In [ ]:
prompt = "Question: Describe this photo. Answer:"
# test_data = train_data[:10]  # 只测试前10条数据，避免一次性处理过多导致显存问题

# ===================== 生成结果 =====================
results = []
for item in test_data:
    # 加载图片
    image = Image.open(item['image']).convert('RGB')
    
    # 生成描述
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=50)
    generated_text = processor.decode(generated_ids[0], skip_special_tokens=True)
    
    # 保存结果（保留原始text）
    results.append({
        'image': item['image'],
        'text': item['text'],
        'generated': generated_text
    })

# ===================== 保存结果 =====================
df = pd.DataFrame(results)
df.to_csv('./blip/test_results.csv', index=False, encoding='utf-8')
print("结果已保存到 test_results.csv")
print(df)

In [ ]:
prompt = "Question: Describe. Answer:"
# test_data = train_data[:10]  # 只测试前10条数据，避免一次性处理过多导致显存问题

# ===================== 生成结果 =====================
results = []
for item in test_data:
    # 加载图片
    image = Image.open(item['image']).convert('RGB')
    
    # 生成描述
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=50)
    generated_text = processor.decode(generated_ids[0], skip_special_tokens=True)
    
    # 保存结果（保留原始text）
    results.append({
        'image': item['image'],
        'text': item['text'],
        'generated': generated_text
    })

# ===================== 保存结果 =====================
df = pd.DataFrame(results)
df.to_csv('./blip/test_resultsc.csv', index=False, encoding='utf-8')
print("结果已保存到 test_resultsc.csv")
print(df)

In [ ]:
prompt = "Question: Describe this photo. Answer:"

# ===================== 生成结果 =====================
results = []
for item in normal_data:
    # 加载图片
    image = Image.open(item['image']).convert('RGB')
    
    # 生成描述
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=50)
    generated_text = processor.decode(generated_ids[0], skip_special_tokens=True)
    
    # 保存结果（保留原始text）
    results.append({
        'image': item['image'],
        'text': item['text'],
        'generated': generated_text
    })

# ===================== 保存结果 =====================
df = pd.DataFrame(results)
df.to_csv('./blip/normal_results.csv', index=False, encoding='utf-8')
print("结果已保存到 normal_results.csv")
print(df)

In [ ]:
prompt = "Question: Describe. Answer:"

# ===================== 生成结果 =====================
results = []
for item in normal_data:
    # 加载图片
    image = Image.open(item['image']).convert('RGB')
    
    # 生成描述
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=50)
    generated_text = processor.decode(generated_ids[0], skip_special_tokens=True)
    
    # 保存结果（保留原始text）
    results.append({
        'image': item['image'],
        'text': item['text'],
        'generated': generated_text
    })

# ===================== 保存结果 =====================
df = pd.DataFrame(results)
df.to_csv('./blip/normal_resultsc.csv', index=False, encoding='utf-8')
print("结果已保存到 normal_resultsc.csv")
print(df)